In [1]:
# @title ##### License { display-mode: "form" }
# Copyright 2019 DeepMind Technologies Ltd. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Zero-Shot LLM Game-Playing Agent with Gemma 4

* This Colab demonstrates using a pre-trained Gemma 4 model as a zero-shot game-playing agent.
* The LLM is prompted to generate a Python policy function for Tic-Tac-Toe.
* The generated policy is then evaluated against a random opponent via [OpenSpiel](https://github.com/google-deepmind/open_spiel).
* No fine-tuning is required — the model produces a working strategy from a single prompt.
* We use the [Hugging Face Transformers](https://huggingface.co/docs/transformers) library to load and run the model.

## Install

In [2]:
# @title Install Dependencies
%pip install -q -U jedi open_spiel transformers accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 175.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 13.1 MB/s eta 0:00:00


## Model Setup

Load the Gemma 4 instruction-tuned model via Hugging Face Transformers.
We use `google/gemma-4-E2B-it` which is small enough to run on a single GPU.

In [29]:
# @title Imports & Model Setup
import json
import re
import numpy as np
import torch
from tqdm import tqdm
import pyspiel
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-4-E2B-it"

print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

print("Hugging Face Gemma 4 is ready")

game = pyspiel.load_game("tic_tac_toe")

Loading google/gemma-4-E2B-it...


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Hugging Face Gemma 4 is ready


## Helper Functions

Utility functions for:
- Extracting Python code from LLM responses (expects markdown-fenced code blocks)
- Generating responses from the Hugging Face model
- Converting OpenSpiel state structs to plain dictionaries

In [30]:
# @title Code extraction, generation, and state conversion helpers
import random
import json
import re
import pyspiel
import numpy as np

game = pyspiel.load_game("tic_tac_toe")


def extract_code(llm_response: str) -> str:
    """Extract Python code from an LLM response.

    Tries ```python fences first, then bare ``` fences.
    Falls back to returning the raw response if no fences are found.
    """
    match = re.search(r'```python\n(.*?)\n```', llm_response, re.DOTALL)
    if match:
        return match.group(1)

    match_any = re.search(r'```\n?(.*?)\n?```', llm_response, re.DOTALL)
    if match_any:
        return match_any.group(1)

    print("Code extraction failed. Raw output:")
    print(llm_response)
    return llm_response


def generate_hf_response(prompt_text: str):
    """Generate a response from the Hugging Face model using greedy decoding."""
    message = [{"role": "user", "content": prompt_text}]

    # Format the prompt using the model's chat template.
    formatted_prompt = tokenizer.apply_chat_template(
        message,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    # Use greedy decoding (do_sample=False) for deterministic output.
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False
    )

    # Decode only the generated tokens (skip the prompt).
    prompt_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
    return response


def struct_to_dict(obj):
    """Convert an OpenSpiel state struct to a plain dictionary.

    Iterates over non-private, non-callable attributes of the struct.
    """
    if isinstance(obj, dict):
        return obj

    out = {}
    for k in dir(obj):
        if k.startswith("_"):
            continue
        v = getattr(obj, k)
        if callable(v):
            continue
        out[k] = v
    return out

## Policy Prompt

The prompt asks the LLM to generate a complete `get_action` function that implements
a simple heuristic strategy for Tic-Tac-Toe: win if possible, block opponent wins,
then prefer center and corners.

In [ ]:
# @title Define the policy prompt

# The prompt instructs the LLM to return code inside a ```python fence
# so that extract_code() can reliably parse it.
POLICY_PROMPT = """
You are an expert game-playing AI programmer.

Your task: write a Python function that plays Tic-Tac-Toe optimally.
The function receives game state from OpenSpiel's state.to_struct() API.

Here is an example of what the inputs look like mid-game:

  state_json = {
      "board": ["x", ".", "o", ".", "x", ".", ".", ".", "."],
      ...
  }
  legal_actions = [1, 3, 5, 6, 7, 8]

- state_json["board"] is a flat list of 9 strings: "x", "o", or ".".
  Indices map to the board as:
    0 | 1 | 2
    ---------
    3 | 4 | 5
    ---------
    6 | 7 | 8
- legal_actions contains the indices of empty cells.
- The current player can be inferred: "x" goes first, so if
  board.count("x") == board.count("o"), it is "x"'s turn.

The 8 winning lines are:
  lines = [[0,1,2],[3,4,5],[6,7,8],[0,3,6],[1,4,7],[2,5,8],[0,4,8],[2,4,6]]

To check if a move wins: place the mark on a copy of the board,
then check if any line has all 3 cells equal to that mark.

To check if a move blocks: check if the OPPONENT would win by
placing THEIR mark at that position.

Write the function inside a ```python code block:

def get_action(state_json, legal_actions):

Write the strongest possible policy.

CRITICAL CONSTRAINTS:
- Do NOT import anything.
- Do NOT include comments or docstrings.
- All logic must be inline inside get_action.
"""


## Generate and Extract the Policy

Send the prompt to Gemma 4 and extract the generated Python function.
The function is then compiled via `exec` and stored as `llm_get_action`.

In [32]:
# @title Generate the policy function from the LLM

response = generate_hf_response(POLICY_PROMPT)
code_str = extract_code(response)

print(code_str)

# Execute the generated code in a sandboxed namespace.
exec_globals = {"random": random}
exec(code_str, exec_globals)

llm_get_action = exec_globals["get_action"]

def get_action(state_json, legal_actions):
    board = state_json["board"]
    current_player = "x" if board.count("x") == board.count("o") else "o"
    opponent = "o" if current_player == "x" else "x"

    def check_win(current_board, player):
        lines = [[0,1,2],[3,4,5],[6,7,8],[0,3,6],[1,4,7],[2,5,8],[0,4,8],[2,4,6]]
        for line in lines:
            if current_board[line[0]] == player and current_board[line[1]] == player and current_board[line[2]] == player:
                return True
        return False

    def check_block(current_board, player):
        for move in legal_actions:
            temp_board = list(current_board)
            temp_board[move] = player
            if check_win(temp_board, opponent):
                return move
        return None

    best_score = -float('inf')
    best_action = legal_actions[0]

    for action in legal_actions:
        temp_board = list(board)
        temp_board[action] = current_player

        if check_win(temp_board, cur

## Evaluate the LLM Policy

Play 100 games of Tic-Tac-Toe: the LLM-generated policy vs. a uniform random opponent.
The LLM player is randomly assigned to play first or second each game.

In [33]:
# @title Evaluate LLM policy vs. random opponent


def safe_llm_policy(state):
    """Wrapper that falls back to a random legal action if the LLM policy fails."""
    raw_struct = state.to_struct()
    state_dict = struct_to_dict(raw_struct)

    legal_actions = state.legal_actions()

    try:
        action = llm_get_action(state_dict, legal_actions)
        if action not in legal_actions:
            return random.choice(legal_actions)
        return action
    except Exception as e:
        print("[!] LLM policy failed:", e)
        return random.choice(legal_actions)


def random_policy_action(state):
    """Uniform random policy: picks a random legal action."""
    return random.choice(state.legal_actions())


def run_evaluation(num_games=100):
    """Run num_games episodes of LLM vs. random and print win/loss/draw stats."""
    wins, losses, draws = 0, 0, 0

    for game_idx in range(num_games):
        state = game.new_initial_state()
        # Randomly assign the LLM to player 0 or 1 each game.
        llm_player = np.random.choice([0, 1])

        while not state.is_terminal():
            current_player = state.current_player()

            if current_player == llm_player:
                action = safe_llm_policy(state)
            else:
                action = random_policy_action(state)

            state.apply_action(action)

        returns = state.returns()

        if returns[llm_player] > 0:
            wins += 1
        elif returns[llm_player] < 0:
            losses += 1
        else:
            draws += 1

    print(f"Results over {num_games} games:")
    print(f"Wins:   {wins}")
    print(f"Losses: {losses}")
    print(f"Draws:  {draws}")
    print(f"Win rate: {(wins / num_games) * 100:.1f}%")
    print(f"Non-loss rate: {((wins + draws) / num_games) * 100:.1f}%")

run_evaluation(num_games=100)

Results over 100 games:
Wins:   79
Losses: 17
Draws:  4
Win rate: 79.0%
Non-loss rate: 83.0%
